In [32]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

In [33]:
df = pd.read_csv("covid_toy.csv")

In [34]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [35]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [36]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [37]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [38]:
from sklearn.model_selection import train_test_split

In [39]:
x_train,x_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),
                                                         df['has_covid'],
                                                         test_size=0.2,
                                                         random_state=0
)

In [40]:
x_train

,age,gender,fever,cough,city
43,22,Female,99.0,Mild,Bangalore
62,56,Female,104.0,Strong,Bangalore
3,31,Female,98.0,Mild,Kolkata
71,75,Female,104.0,Strong,Delhi
45,72,Male,99.0,Mild,Bangalore
...,...,...,...,...,...
96,51,Female,101.0,Strong,Kolkata
67,65,Male,99.0,Mild,Bangalore
64,42,Male,104.0,Mild,Mumbai
47,18,Female,104.0,Mild,Bangalore


# Without column Transformer class

In [41]:
si = SimpleImputer()


In [42]:
x_train_fever = si.fit_transform(x_train[['fever']])
x_test_fever = si.fit_transform(x_test[['fever']])

In [43]:
x_train.head()

,age,gender,fever,cough,city
43,22,Female,99.0,Mild,Bangalore
62,56,Female,104.0,Strong,Bangalore
3,31,Female,98.0,Mild,Kolkata
71,75,Female,104.0,Strong,Delhi
45,72,Male,99.0,Mild,Bangalore


In [44]:
x_train_fever.shape

(80, 1)

# OrdinalEncoding

In [45]:
Ordinal = OrdinalEncoder(categories=[['Mild' , 'Strong']])

x_train_cough = Ordinal.fit_transform(x_train[['cough']])

x_test_cough = Ordinal.fit_transform(x_test[['cough']])


In [46]:
x_train_cough.shape

(80, 1)

# OneHot Encoding

In [47]:
OE = OneHotEncoder(drop='first' , sparse_output=False) # drop first for multicollinearlity

x_train_gender_city = OE.fit_transform(x_train[['gender' , 'city']])


x_test_gender_city = OE.fit_transform(x_test[['gender' , 'city']])


In [48]:
x_train_gender_city.shape

(80, 4)

# Extracting Age

In [49]:
x_train_age = x_train.drop(columns = ['gender' ,'fever', 'cough','city']).values
x_test_age = x_test.drop(columns = ['gender' ,'fever', 'cough','city']).values

In [50]:
x_train_age.shape

(80, 1)

In [51]:
x_train_transformed = np.concatenate((x_train_age , x_train_fever, x_train_gender_city,x_train_cough),axis=1)
x_test_transformed = np.concatenate((x_test_age,x_test_fever,x_test_gender_city,x_test_cough),axis=1)

In [ ]:
x_train_transformed = pd.DataFrame(x_train_transformed)

In [53]:
x_train_transformed

,0,1,2,3,4,5,6
0,22.0,99.0,0.0,0.0,0.0,0.0,0.0
1,56.0,104.0,0.0,0.0,0.0,0.0,1.0
2,31.0,98.0,0.0,0.0,1.0,0.0,0.0
3,75.0,104.0,0.0,1.0,0.0,0.0,1.0
4,72.0,99.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...
75,51.0,101.0,0.0,0.0,1.0,0.0,1.0
76,65.0,99.0,1.0,0.0,0.0,0.0,0.0
77,42.0,104.0,1.0,0.0,0.0,1.0,0.0
78,18.0,104.0,0.0,0.0,0.0,0.0,0.0


# With Sklearn Column Transformer

In [54]:
from sklearn.compose import ColumnTransformer

In [55]:
x_train

,age,gender,fever,cough,city
43,22,Female,99.0,Mild,Bangalore
62,56,Female,104.0,Strong,Bangalore
3,31,Female,98.0,Mild,Kolkata
71,75,Female,104.0,Strong,Delhi
45,72,Male,99.0,Mild,Bangalore
...,...,...,...,...,...
96,51,Female,101.0,Strong,Kolkata
67,65,Male,99.0,Mild,Bangalore
64,42,Male,104.0,Mild,Mumbai
47,18,Female,104.0,Mild,Bangalore


In [69]:
transformer =  ColumnTransformer( transformers=[  
    ('tnf1',SimpleImputer() ,['fever']), # We pass transformers in tuple  and in tupple we pass (name , Transformation , [column])
    ('tnf2',OneHotEncoder(drop='first' , sparse_output=False,    dtype=np.int32),['gender' , 'city']),
    ('tnf3',OrdinalEncoder(categories=[['Mild' , 'Strong']],    dtype=np.int32),['cough'])],
      remainder='passthrough',#reminder has to Drop and passthrough remaining columns
      verbose_feature_names_out=False

    )  

In [70]:
pd.DataFrame(transformer.fit_transform(x_train).astype(int),columns=transformer.get_feature_names_out()) #Getfeature name out for  Columns name

,fever,gender_Male,city_Delhi,city_Kolkata,city_Mumbai,cough,age
0,99,0,0,0,0,0,22
1,104,0,0,0,0,1,56
2,98,0,0,1,0,0,31
3,104,0,1,0,0,1,75
4,99,1,0,0,0,0,72
...,...,...,...,...,...,...,...
75,101,0,0,1,0,1,51
76,99,1,0,0,0,0,65
77,104,1,0,0,1,0,42
78,104,0,0,0,0,0,18


In [58]:
transformer.fit_transform(x_test).shape

(20, 7)